# Data Cleaning
ทำความสะอาดข้อมูลจาก 3 ไฟล์หลัก แล้ว export เป็น `cleaned/` folder:
- `master_results_cleaned.csv`
- `master_summary_cleaned.csv`
- `coords_cleaned.csv` (ใช้ output_with_coordinates แทน unit_position / unit_village)

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('../')
OUT  = Path('cleaned/')
OUT.mkdir(exist_ok=True)

results  = pd.read_csv(BASE / 'master_results.csv')
summary  = pd.read_csv(BASE / 'master_summary.csv', index_col=0)
coords   = pd.read_csv(BASE / 'output_with_coordinates.csv', index_col=0)

print('Loaded:')
print(f'  results : {results.shape}')
print(f'  summary : {summary.shape}')
print(f'  coords  : {coords.shape}')

Loaded:
  results : (22422, 7)
  summary : (710, 10)
  coords  : (342, 9)


---
## 1. Clean master_results.csv

### 1.1 แก้ค่า type ผิด: 'บч' → 'บช'

In [24]:
r = results.copy()

before = (r['type'] == 'บч').sum()
r['type'] = r['type'].replace('บч', 'บช')
print(f'แก้ type บч → บช: {before} rows')
print('type unique หลังแก้:', r['type'].unique())

แก้ type บч → บช: 1 rows
type unique หลังแก้: ['บช' 'เขต']


### 1.2 กรองเฉพาะ province == 'ลำปาง'

In [25]:
noise_provinces = r[r['province'] != 'ลำปาง']['province'].value_counts()
print('province noise ที่จะถูกลบ:')
print(noise_provinces)

before = len(r)
r = r[r['province'] == 'ลำปาง'].copy()
print(f'\nลบ noise province: {before} → {len(r)} rows ({before - len(r)} ลบออก)')

province noise ที่จะถูกลบ:
province
สีคาม           57
สุโขทัย         57
สุราษฎร์ธานี    15
Name: count, dtype: int64

ลบ noise province: 22422 → 22293 rows (129 ลบออก)


In [26]:
### 1.2b แก้ชื่ออำเภอ (district) ที่ OCR เพี้ยน
DISTRICT_MAP = {
    'งาม':             'งาว',
    'นางา':            'งาว',   # OCR เพี้ยน ตำบลนาแก unit 4 อยู่ในอำเภองาว
    'องาว':            'งาว',
    'อำเภองาว':        'งาว',
    'อำเภอวังเหนือ':   'วังเหนือ',
    'อำเภอเมืองปาน':   'เมืองปาน',
    'อำเภอเมืองลำปาง': 'เมืองลำปาง',
    'นางลิ้นจี่':      'แจ้ห่ม',   # OCR เพี้ยน ตำบลนางลิ้นจี่อยู่ในอำเภอแจ้ห่ม
}

before_d = r['district'].nunique()
r['district'] = r['district'].replace(DISTRICT_MAP)
after_d = r['district'].nunique()
print(f'district unique: {before_d} → {after_d}')
print('district ทั้งหมด:', sorted(r['district'].unique()))

district unique: 15 → 7
district ทั้งหมด: ['งาว', 'นอกเขต', 'นางเงาน', 'วังเหนือ', 'เมืองปาน', 'เมืองลำปาง', 'แจ้ห่ม']


### 1.3 แปลง score เป็นตัวเลข และลบแถวผิดปกติ

In [27]:
r['score'] = pd.to_numeric(r['score'], errors='coerce')

# score = -1 → missing value (เหมือน pattern ใน summary)
neg_one = r[r['score'] == -1]
print(f'score == -1 (missing): {len(neg_one)} rows')
print(neg_one['type'].value_counts())

# score < 0 แต่ไม่ใช่ -1
other_neg = r[(r['score'] < 0) & (r['score'] != -1)]
print(f'\nscore < 0 อื่นๆ: {len(other_neg)} rows')
if len(other_neg) > 0:
    print(other_neg[['district','sub-district','unit_number','name','score']])

score == -1 (missing): 1662 rows
type
บช     1526
เขต     136
Name: count, dtype: int64

score < 0 อื่นๆ: 13 rows
       district sub-district  unit_number                         name  score
10032  วังเหนือ       วังทอง            2                 ทางเลือกใหม่   -3.0
10033  วังเหนือ       วังทอง            2                     เศรษฐกิจ   -3.0
10035  วังเหนือ       วังทอง            2               รวมพลังประชาชน   -4.0
10038  วังเหนือ       วังทอง            2                 พลังเพื่อไทย   -4.0
10039  วังเหนือ       วังทอง            2                       ไทยชนะ   -2.0
10044  วังเหนือ       วังทอง            2                     ก้าวล่วง   -2.0
10046  วังเหนือ       วังทอง            2                    วิชัยใหม่   -4.0
10049  วังเหนือ       วังทอง            2                 ประชาธิปัตย์   -9.0
10051  วังเหนือ       วังทอง            2                     ไทยภักดี   -2.0
10054  วังเหนือ       วังทอง            2           ครูไทยเพื่อประชาชน   -3.0
10063  วังเหนือ       วังทอง

In [28]:
# score > 99999 → OCR ต่อเลขกัน
bad_high = r[r['score'] > 99999]
print(f'score > 99999 (OCR ต่อเลข): {len(bad_high)} rows')
print(bad_high[['district','sub-district','unit_number','name','score']])

# score มีทศนิยม (OCR อ่านเลขผิด เช่น 72.80, 0.50)
bad_frac = r[(r['score'] > 0) & (r['score'] % 1 != 0) & (r['unit_number'] != -1)]
print(f'\nscore มีทศนิยมผิดปกติ: {len(bad_frac)} rows')
print(bad_frac[['district','sub-district','unit_number','name','score']])

score > 99999 (OCR ต่อเลข): 13 rows
       district sub-district  unit_number             name      score
7406   วังเหนือ     ทุ่งฮั้ว            6         รวมใจไทย   200000.0
8380   วังเหนือ     ร่องเคาะ           18         เศรษฐกิจ  7000000.0
8381   วังเหนือ     ร่องเคาะ           18       เสรีรวมไทย  3000000.0
8382   วังเหนือ     ร่องเคาะ           18   รวมพลังประชาชน  1000000.0
8385   วังเหนือ     ร่องเคาะ           18     พลังเพื่อไทย  1000000.0
8386   วังเหนือ     ร่องเคาะ           18           ไทยชนะ  2000000.0
8391   วังเหนือ     ร่องเคาะ           18        ก้าวลิสระ  1000000.0
8393   วังเหนือ     ร่องเคาะ           18       วิชช์นใหม่  1000000.0
8394   วังเหนือ     ร่องเคาะ           18   เพื่อชีวิตใหม่  1000000.0
8395   วังเหนือ     ร่องเคาะ           18          คลองไทย  1000000.0
8396   วังเหนือ     ร่องเคาะ           18     ประชาธิปัตย์  3900000.0
8399   วังเหนือ     ร่องเคาะ           18  แรงงานสร้างชาติ  1000000.0
13316  เมืองปาน     หัวเมือง            4          ประ

In [29]:
before = len(r)

# ลบ: score < 0 (รวม -1) และ score > 99999
r = r[(r['score'] >= 0) & (r['score'] <= 99999)].copy()

# ปัดทศนิยมที่เกิดจาก OCR → round เป็น int
r['score'] = r['score'].round().astype('Int64')

print(f'ลบแถวผิดปกติ: {before} → {len(r)} rows (ลบ {before - len(r)} rows)')
print('score dtype:', r['score'].dtype)
print('score min:', r['score'].min(), '| max:', r['score'].max())

ลบแถวผิดปกติ: 22293 → 20600 rows (ลบ 1693 rows)
score dtype: Int64
score min: 0 | max: 10290


### 1.4 Normalize ชื่อผู้สมัคร ส.ส. เขต (Entity Resolution)

In [30]:
# ผู้สมัครจริงมี 7 คน ที่เหลือเป็น OCR เพี้ยน
CANDIDATE_MAP = {
    # นายดาซัย เอกปฐพี
    'นายดาชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายตาซัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาซัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายดาชาย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายตาชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดำชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาชัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายตาซัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายดาซัย เออปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาซัย เอกปรูพี':  'นายดาซัย เอกปฐพี',

    # นางสาวสุวิภา กุศลจูง
    'นางสาวสุวิภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลงู':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลถุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลอุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุคลุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กูคลุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กฤตจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวภา กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาวิภา กุศลจูง':  'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาว กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจูง':      'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจุง':      'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจรุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิวา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภภา กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภารา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุชิภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุรภา กุศลจุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุริภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุริภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นายสารสุภา กุคลจรุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาวิกฤตจูง':      'นางสาวสุวิภา กุศลจูง',

    # นายธนาธร โล่ห์สุนทร
    'นายธนาธร โล่หลุนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โล่หลุ่นทร':  'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร ไล่ห์สุนทร':  'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โสฬสุนทร':    'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โลห์สุนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โล่ห์สนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร ไล่หุ่นทร':   'นายธนาธร โล่ห์สุนทร',

    # นางสาววิชุดา ว่องวัฒนวิโรจน์
    'นางสาววิชุดา ว่องวัฒน์โรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนาโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนาวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒน์วโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒน์วิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวันวนิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวันณวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่วงวัฒน์วโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา อ่วงวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชา ว่องวัฒนวิโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชา ว่องวัฒนวโรจน์':      'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชิตา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชาดา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชาดา ว่องวัฒนวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชาดา ว่องวัฒนาโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูชา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒน์โรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒน์วิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาวรวิชุดา ว่องวัฒนวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาวรวิชดา ว่องวัฒนวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฤดา วงศ์พรมไพรณ์':        'นางสาววิชุดา ว่องวัฒนวิโรจน์',

    # พันเอกสันทัด ภัทรกิตตินนท์
    'พันเอกสันทัด ภัทรกิตตินันท์':  'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภักรกิตตินนท์':   'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภัทรคิตตินนท์':   'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภัทรเกิดตินนท์':  'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสืบทัต ภัทรเกิดสินท์':   'พันเอกสันทัด ภัทรกิตตินนท์',

    # นายสมบูรณ์ รูปสะอาด
    'นายสมบูรณ์ รูปละอาด': 'นายสมบูรณ์ รูปสะอาด',

    # นายศรีพรหม หอมยก
    'นายศรีพรหมอ หอมยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอ มอมยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอ มอยยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรรณ หอมเอก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอภย':       'นายศรีพรหม หอมยก',
}

mask_ket = r['type'] == 'เขต'
before_ket = r[mask_ket]['name'].nunique()
r.loc[mask_ket, 'name'] = r.loc[mask_ket, 'name'].replace(CANDIDATE_MAP)
after_ket = r[mask_ket]['name'].nunique()
print(f'เขต unique name: {before_ket} → {after_ket}')
print('ผู้สมัครที่เหลือ:', sorted(r[mask_ket]['name'].unique()))

เขต unique name: 145 → 70
ผู้สมัครที่เหลือ: ['ดร.วิชา ฮ่องวิมเนเจอร์', 'น.ส. วิชดา ว่องอัฒนาโรจน์', 'น.ส. ศุภิกา กุศลจรูญ', 'น.ส. สวิภา กุศลคง', 'น.ส.วิชาดา วงษ์มนวโรจน์ ประชาธิปัตย์', 'น.ส.วิธูภา ล่องวัฒนาโรจน์', 'น.ส.สาวภา กศฤทธุง', 'นส.สุจิต กุมลพูน', 'นักเอกสันทัด ภัทรสถิติแพร', 'นันเอก สืบกด ภัทธากิตติกานต์', 'นางสมบูรณ์ ธูปสะอาด', 'นางสาว กิตวิทยานิโรธ', 'นางสาวธิดารดา วงศ์อนันต์โสน', 'นางสาววิชุดา วิญวัฒนวรโรจน์', 'นางสาววิชุดา ว่องวัฒนวิโรจน์', 'นางสาววิธยา ก่อนจำนำโภชน์', 'นางสาววิภา วงศ์พันธ์โรจน์', 'นางสาววิฤดา ว่องวัฒน์โรจน์', 'นางสาวศรวภา กุศลคง', 'นางสาวสร้างกา กุศลดุง', 'นางสาวสุวิตา กุศลกูฏ', 'นางสาวสุวิภา กุศลจูง', 'นาย ศรีพรหม หอมผกก', 'นายคมพูรณ์ รูปสะอาด', 'นายณัฐพรหม พรมมา', 'นายดาชนัย เอกประที', 'นายดาชัย แวกประที', 'นายดาซัย เอกปฐพี', 'นายดารัชย์ แอกประที', 'นายดารัย เอกปฐมพี', 'นายดาริช เฉลาประที', 'นายดำชัย เอกประพันธ์', 'นายดำรงค์ พรมมา', 'นายธนาธร โกสินทร์เพ็ญ', 'นายธนาธร โถ่ห์สันทร', 'นายธนาธร โลหะไพบูลย์', 'นายธนาธร โล่พลันทร', 'นายธนาธร โล่พันธุ์ทร', 'นายธ

In [31]:
# Dictionary mapping ชื่อ OCR เพี้ยน → ชื่อมาตรฐาน
PARTY_MAP = {
    # พรรคประชาชน (ชื่อเดิมก่อนถูกยุบ: ก้าวไกล — ตั้งใหม่เป็นพรรคประชาชน ปี 67)
    'ก้าวไกล': 'พรรคประชาชน',
    # พรรคประชาชน / วิชชั่นใหม่ (Vision New)
    'ก้าวลิสระ': 'พรรคประชาชน', 'ก้าวลิกระยะ': 'พรรคประชาชน', 'ก้าวลิตระ': 'พรรคประชาชน',
    'ก้าวลิตรระ': 'พรรคประชาชน', 'ก้าวลิตรสะ': 'พรรคประชาชน', 'ก้าวลิตราะ': 'พรรคประชาชน',
    'ก้าวลิละระ': 'พรรคประชาชน', 'ก้าวลิสง': 'พรรคประชาชน', 'ก้าวลิสงระ': 'พรรคประชาชน',
    'ก้าวลิสรละ': 'พรรคประชาชน', 'ก้าวลิสรสะ': 'พรรคประชาชน', 'ก้าวลิ่วสระ': 'พรรคประชาชน',
    'ก้าวล่วง': 'พรรคประชาชน', 'ก้าวอิสระ': 'พรรคประชาชน', 'ก้าวก้มลง': 'พรรคประชาชน',
    'ไทยก้าวใหม่': 'พรรคประชาชน', 'ไทยกว้าใหม่': 'พรรคประชาชน', 'ไทยกว่าใหม่': 'พรรคประชาชน',
    'ไทยกว้างใหม่': 'พรรคประชาชน', 'ไทยกว้าไหม': 'พรรคประชาชน', 'ไทยกว้าใหญ่': 'พรรคประชาชน',
    'ไทยกว้างใหญ่': 'พรรคประชาชน', 'ไทยกว้านใหม่': 'พรรคประชาชน', 'ไทยกว้าวใหม่': 'พรรคประชาชน',
    'ไทยกาวใหม่': 'พรรคประชาชน',

    # วิชชั่นใหม่
    'วิชช์นใหม่': 'วิชชั่นใหม่', 'วิชชินใหม่': 'วิชชั่นใหม่', 'วิชชิมใหม่': 'วิชชั่นใหม่',
    'วิชชิ่นใหม่': 'วิชชั่นใหม่', 'วิชช์ขึ้นใหม่': 'วิชชั่นใหม่', 'วิชช์ชั่นใหม่': 'วิชชั่นใหม่',
    'วิชช์ใหม่': 'วิชชั่นใหม่', 'วิชชันใหม่': 'วิชชั่นใหม่', 'วิชัยขับใหม่': 'วิชชั่นใหม่',
    'วิชัยขั้นใหม่': 'วิชชั่นใหม่', 'วิชัยขึ้นใหม่': 'วิชชั่นใหม่', 'วิชัยชั้นใหม่': 'วิชชั่นใหม่',
    'วิชัยซันใหม่': 'วิชชั่นใหม่', 'วิชัยใหม่': 'วิชชั่นใหม่', 'วิชาชีพใหม่': 'วิชชั่นใหม่',
    'วิชัยขับใหญ่': 'วิชชั่นใหม่', 'ภิติใหม่': 'วิชชั่นใหม่',

    # สร้างอนาคตไทย
    'สร้างอนาคคไทย': 'สร้างอนาคตไทย', 'สรางอนาคตไทย': 'สร้างอนาคตไทย',
    'สรางอนาคไทย': 'สร้างอนาคตไทย', 'สร้างอนาคตกไทย': 'สร้างอนาคตไทย',
    'สร้างอนาคไทย': 'สร้างอนาคตไทย',

    # ไทยก้าวหน้า
    'ไทยก้าวหน้าแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นาแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นำแนวประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นำแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า มาแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า แห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้านานดีประเทศไทย': 'ไทยก้าวหน้า',

    # ไทยชนะ
    'ไทยชนะ รวม': 'ไทยชนะ', 'ไทยชนะรวม': 'ไทยชนะ', 'ไทมชนะ': 'ไทยชนะ',

    # ไทยภักดี
    'ไทยภักดี ธรรม': 'ไทยภักดี',

    # ไทยพร้อม
    'ไทยพร้อม -': 'ไทยพร้อม',

    # ไทยพิทักษ์ธรรม
    'ไทยพิทักษ์กรรม': 'ไทยพิทักษ์ธรรม',

    # ไทยทรัพย์ทวี
    'ไทยทรัพย์ไว': 'ไทยทรัพย์ทวี', 'ไทยทรัพยวิถี': 'ไทยทรัพย์ทวี',
    'ไทยทรัพยากร': 'ไทยทรัพย์ทวี',

    # ไทยรวมไทย
    'ไทรวมพลัง': 'ไทยรวมไทย', 'ไทไรวมพลัง': 'ไทยรวมไทย',

    # ประชาธิปัตย์
    'ประชาธิปโดยใหม่': 'ประชาธิปัตย์', 'ประชาธิปไตยใหม่': 'ประชาธิปัตย์',
    'ประชาธิปัตย์ใหม่': 'ประชาธิปัตย์', 'ประชาธิปโดยใหม่': 'ประชาธิปัตย์',
    'ประชาชาติปัตย์': 'ประชาธิปัตย์', 'ใหม่ราชิติปัตย์': 'ประชาธิปัตย์',

    # ประชาไทย
    'ประชนไทย': 'ประชาไทย', 'ประชาอาสาชาติ': 'ประชาไทย',

    # ครูไทยเพื่อประชาชน
    'ครูไทยเทือประชาน': 'ครูไทยเพื่อประชาชน',

    # คลองไทย
    'คลองไทย สภาชาติ': 'คลองไทย', 'คลองไทยชาติ': 'คลองไทย',

    # เพื่อชาติไทย
    'เพื่อชาติใหม่': 'เพื่อชาติไทย', 'เพื่อชาติติไทย': 'เพื่อชาติไทย',
    'เพื่อชาติต่อไป': 'เพื่อชาติไทย',

    # เพื่อชีวิตใหม่
    'เพื่อขีวิตใหม่': 'เพื่อชีวิตใหม่', 'เทื่อชีวิตใหม่': 'เพื่อชีวิตใหม่',

    # เพื่อบ้านเมือง
    'เทือบ้านเมือง': 'เพื่อบ้านเมือง',

    # พลังไทยรักชาติ
    'พลังไทyrักชาติ': 'พลังไทยรักชาติ', 'พลังไทธรักชาติ': 'พลังไทยรักชาติ',
    'หลังไทยรักชาติ': 'พลังไทยรักชาติ',

    # พลวัต
    'พลวัด': 'พลวัต',

    # พิวจันทร์ (ถ้ามี)
    'พิวจัน': 'พิวจันทร์', 'พิวซัน': 'พิวจันทร์', 'พิวซันใหม่': 'พิวจันทร์',
    'พิวชัน': 'พิวจันทร์', 'พิวซิน': 'พิวจันทร์', 'พิวัฒน์': 'พิวจันทร์',
    'พิวขัน': 'พิวจันทร์', 'พิวจัน': 'พิวจันทร์', 'ฟิวชัน': 'พิวจันทร์',
    'ฟิวซัน': 'พิวจันทร์', 'พิกขัน': 'พิวจันทร์', 'พิกวัน': 'พิวจันทร์',
    'พิกุล': 'พิวจันทร์', 'พิกุลชน': 'พิวจันทร์', 'พิกุลัน': 'พิวจันทร์',
    'พิมพ์ขึ้น': 'พิวจันทร์', 'พิวจันทร์': 'พิวจันทร์', 'พิวซัน ใหม่': 'พิวจันทร์',
    'พืชขัน': 'พิวจันทร์', 'พืชชน': 'พิวจันทร์', 'พืชชัน': 'พิวจันทร์',
    'พืชซัน': 'พิวจันทร์', 'ฟิลิปปิน': 'พิวจันทร์',

    # รักชาติ
    'วักชาติ': 'รักชาติ',

    # รวมไทยสร้างชาติ
    'รวมใจไทย': 'รวมไทยสร้างชาติ',

    # ท้องที่ไทย
    'ห้องที่ไทย': 'ท้องที่ไทย', 'พ้องที่ไทย': 'ท้องที่ไทย',
    'ท้องที่ไทยใหม่': 'ท้องที่ไทย',

    # สังคมประชาธิปไตยไทย
    'สังคมประชาธิปโดยไทย': 'สังคมประชาธิปไตยไทย',

    # เครือข่ายชาวนาแห่งประเทศไทย
    'เครือขายชาวนาแห่งประเทศไทย': 'เครือข่ายชาวนาแห่งประเทศไทย',

    # ใหม่ (พรรคกรีน)
    'กรีน': 'กรีน', 'กริน': 'กรีน',

    # noise / garbage rows
    'รวมคะแนนทั้งสิ้น': None,
}

before_unique = r['name'].nunique()
# apply เฉพาะ บช rows เพื่อไม่ให้ชื่อผู้สมัคร (เขต) โดน overwrite
mask_bch = r['type'] == 'บช'
r.loc[mask_bch, 'name'] = r.loc[mask_bch, 'name'].replace(PARTY_MAP)

# ลบแถวที่ map เป็น None (garbage) — เฉพาะใน บช rows
garbage = mask_bch & r['name'].isna()
print(f'ลบ garbage name rows: {garbage.sum()}')
r = r[~garbage].copy()

after_unique = r['name'].nunique()
print(f'unique name: {before_unique} → {after_unique}')


ลบ garbage name rows: 1
unique name: 245 → 138


### 1.5 แยก 'นอกเขต' ออกเป็น subset

In [32]:
r_outside = r[r['unit_number'] == -1].copy()
r_inzone  = r[r['unit_number'] != -1].copy()

print(f'ในเขต  : {len(r_inzone):,} rows')
print(f'นอกเขต : {len(r_outside):,} rows')

ในเขต  : 19,948 rows
นอกเขต : 651 rows


### 1.6 Cross-validation: score_sum vs valid_ballots

In [33]:
s_tmp = summary.copy()
s_tmp['type'] = s_tmp['type'].replace('บч', 'บช')
s_tmp = s_tmp[s_tmp['province'] == 'ลำปาง']
ballot_cols = ['total_ballots','valid_ballots','invalid_ballots','no_vote_ballots','remain_ballots']
for col in ballot_cols:
    s_tmp[col] = s_tmp[col].replace(-1, np.nan)
s_tmp = s_tmp.dropna(subset=['total_ballots'])
s_in = s_tmp[s_tmp['unit_number'] != -1]

JOIN = ['type','district','sub-district','unit_number']
score_sum = r_inzone.groupby(JOIN)['score'].sum().reset_index()
score_sum.columns = JOIN + ['score_sum']

cv = score_sum.merge(s_in[JOIN + ['valid_ballots']], on=JOIN, how='inner')
cv['diff'] = cv['score_sum'] - cv['valid_ballots']

outliers = cv[cv['diff'] > 500].sort_values('diff', ascending=False)
print(f'หน่วยที่ score_sum เกิน valid_ballots >500: {len(outliers)} units')
print(outliers[JOIN + ['score_sum','valid_ballots','diff']].to_string())

หน่วยที่ score_sum เกิน valid_ballots >500: 32 units
    type    district sub-district  unit_number  score_sum  valid_ballots     diff
204   บช    เมืองปาน     หัวเมือง            4      53062          226.0  52836.0
147   บช    วังเหนือ       วังทอง            7      27897          432.0  27465.0
108   บช    วังเหนือ     ร่องเคาะ            5      10394          199.0  10195.0
533  เขต    เมืองปาน     หัวเมือง            3       7996          196.0   7800.0
423  เขต    วังเหนือ     ทุ่งฮั้ว            1       7493          271.0   7222.0
94    บช    วังเหนือ     ทุ่งฮั้ว            7       6523          499.0   6024.0
209   บช    เมืองปาน     หัวเมือง            9       4745          165.0   4580.0
52    บช         งาว     บ้านโป่ง           10       4514          279.0   4235.0
121   บช    วังเหนือ     ร่องเคาะ           18       4179          242.0   3937.0
244   บช  เมืองลำปาง    บ้านเสด็จ           10       3940          200.0   3740.0
427  เขต    วังเหนือ     ทุ่งฮั้ว            

In [34]:
# ลบ outlier units ที่ diff > 500 ออกจาก r_inzone (OCR ยังหลุดอยู่)
bad_units = outliers[['type','district','sub-district','unit_number']].drop_duplicates()
bad_key = bad_units.apply(tuple, axis=1).tolist()
r_key = r_inzone[['type','district','sub-district','unit_number']].apply(tuple, axis=1)

before = len(r_inzone)
r_inzone = r_inzone[~r_key.isin(bad_key)].copy()
r_outside_orig = r[r['unit_number'] == -1].copy()
r = pd.concat([r_inzone, r_outside_orig], ignore_index=True)

print(f'ลบ outlier units: {before} → {len(r_inzone)} rows ในเขต (ลบ {before - len(r_inzone)} rows)')
print(f'r ทั้งหมด: {len(r):,} rows')

ลบ outlier units: 19948 → 18849 rows ในเขต (ลบ 1099 rows)
r ทั้งหมด: 19,500 rows


### 1.7 ตรวจสอบผลลัพธ์

In [35]:
print('dtype score:', r['score'].dtype)
print('\nสถิติ score (ทุก row ที่เหลือ):')
print(r['score'].describe())
print('\nTop 15 คะแนนรวม (ในเขต, บช):')
top = r_inzone[r_inzone['type']=='บช'].groupby('name')['score'].sum().sort_values(ascending=False).head(15)
print(top.to_string())

dtype score: Int64

สถิติ score (ทุก row ที่เหลือ):
count     19500.0
mean     9.724308
std      41.93637
min           0.0
25%           0.0
50%           1.0
75%           4.0
max        4000.0
Name: score, dtype: Float64

Top 15 คะแนนรวม (ในเขต, บช):
name
ประชาชน            23759
เพื่อไทย           20527
กล้าธรรม            7493
ภูมิใจไทย           5550
รวมไทยสร้างชาติ     5504
ประชาธิปัตย์        3631
เพื่อชาติไทย        3461
เศรษฐกิจ            2539
พลังไทยรักชาติ      1434
ใหม่                1030
รวมพลังประชาชน       987
ไทยทรัพย์ทวี         975
พลังประชารัฐ         726
เสรีรวมไทย           712
ประชาไทย             641


### 1.8 Export

In [36]:
r.to_csv(OUT / 'master_results_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved master_results_cleaned.csv:', r.shape)

Saved master_results_cleaned.csv: (19500, 7)


---
## 2. Clean master_summary.csv

### 2.1 แก้ type และ province noise

In [37]:
s = summary.copy()

s['type'] = s['type'].replace('บч', 'บช')
s['district'] = s['district'].replace(DISTRICT_MAP)  # normalize district names (same as results)

noise_prov = s[s['province'] != 'ลำปาง']['province'].value_counts()
if len(noise_prov) > 0:
    print('province noise:', noise_prov.to_dict())
s = s[s['province'] == 'ลำปาง'].copy()

print('type unique:', s['type'].unique())
print('province unique:', s['province'].unique())

type unique: ['บช' 'เขต']
province unique: ['ลำปาง']


### 2.2 จัดการค่า -1 (Missing Value)

In [38]:
ballot_cols = ['total_ballots','valid_ballots','invalid_ballots','no_vote_ballots','remain_ballots']

mask_neg = (s[ballot_cols] == -1).any(axis=1)
print(f'แถวที่มีค่า -1: {mask_neg.sum()} rows')
print(s[mask_neg][['type','district','sub-district','unit_number'] + ballot_cols])

for col in ballot_cols:
    s[col] = s[col].replace(-1, np.nan)

before = len(s)
s = s.dropna(subset=['total_ballots']).copy()
print(f'\nDrop แถว total_ballots=NaN: {before} → {len(s)} rows (ลบ {before - len(s)} rows)')

แถวที่มีค่า -1: 30 rows
    type    district sub-district  unit_number  total_ballots  valid_ballots  \
0     บช      นอกเขต     ชุดที่ 1           -1            700            663   
1    เขต      นอกเขต     ชุดที่ 1           -1            700            605   
2     บช      นอกเขต    ชุดที่ 10           -1            700            668   
3    เขต      นอกเขต    ชุดที่ 10           -1            700            616   
4     บช      นอกเขต    ชุดที่ 11           -1            700            664   
5    เขต      นอกเขต    ชุดที่ 11           -1            700            607   
6     บช      นอกเขต    ชุดที่ 12           -1            700            666   
7    เขต      นอกเขต    ชุดที่ 12           -1            700            614   
8     บช      นอกเขต    ชุดที่ 13           -1            694            662   
9    เขต      นอกเขต    ชุดที่ 13           -1            694            599   
10    บช      นอกเขต     ชุดที่ 2           -1            700            655   
11   เขต      นอ

### 2.3 แยก 'นอกเขต' และตรวจสอบ

In [39]:
s_outside = s[s['unit_number'] == -1].copy()
s_inzone  = s[s['unit_number'] != -1].copy()

print(f'ในเขต  : {len(s_inzone)} rows')
print(f'นอกเขต : {len(s_outside)} rows')
print('\nสถิติหลัง clean:')
print(s[ballot_cols].describe())

ในเขต  : 681 rows
นอกเขต : 26 rows

สถิติหลัง clean:
       total_ballots  valid_ballots  invalid_ballots  no_vote_ballots  \
count     707.000000     706.000000       706.000000       706.000000   
mean      432.599717     280.794618        20.745042        12.957507   
std       145.432410     114.460345        10.472260        10.737270   
min        80.000000      49.000000         0.000000         0.000000   
25%       330.000000     208.000000        13.000000         6.000000   
50%       420.000000     266.000000        20.000000        11.000000   
75%       520.000000     333.000000        27.000000        16.000000   
max       760.000000     677.000000        66.000000        69.000000   

       remain_ballots  
count      680.000000  
mean       122.342647  
std         48.716897  
min         23.000000  
25%         87.000000  
50%        116.000000  
75%        150.000000  
max        309.000000  


### 2.4 Export

In [40]:
s.to_csv(OUT / 'master_summary_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved master_summary_cleaned.csv:', s.shape)

Saved master_summary_cleaned.csv: (707, 10)


---
## 3. Clean coords (output_with_coordinates.csv)

In [41]:
c = coords.copy()
print('columns:', list(c.columns))
print('null count:')
print(c.isnull().sum())

has_coord = c[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'\nมีพิกัด: {has_coord}/{len(c)} ({has_coord/len(c)*100:.1f}%)')

dup_coord = c[c.duplicated(subset=['latitude','longitude'], keep=False)]
print(f'พิกัดซ้ำกัน (หลายหน่วยใช้จุดเดียว): {len(dup_coord)} rows')
print('  → ไม่ลบทิ้ง เพราะปกติสำหรับพิกัดระดับหมู่บ้าน, ใช้ jitter ตอนพล็อตแผนที่')

columns: ['province', 'district', 'sub-district', 'unit_number', 'village', 'place', 'latitude', 'longitude', 'full_address']
null count:
province        0
district        0
sub-district    0
unit_number     0
village         0
place           0
latitude        0
longitude       0
full_address    0
dtype: int64

มีพิกัด: 342/342 (100.0%)
พิกัดซ้ำกัน (หลายหน่วยใช้จุดเดียว): 187 rows
  → ไม่ลบทิ้ง เพราะปกติสำหรับพิกัดระดับหมู่บ้าน, ใช้ jitter ตอนพล็อตแผนที่


In [42]:
c.to_csv(OUT / 'coords_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved coords_cleaned.csv:', c.shape)

Saved coords_cleaned.csv: (342, 9)


---
## 4. สรุป Cleaning Report

In [43]:
print('=' * 55)
print('CLEANING REPORT')
print('=' * 55)
print(f'master_results : {len(results):,} → {len(r):,} rows  (ลบ {len(results)-len(r):,} rows)')
print(f'  - แก้ type บч → บช')
print(f'  - ลบ province noise')
print(f'  - ลบ score < 0 (missing) และ score > 99999 (OCR ต่อเลข)')
print(f'  - ปัดทศนิยม OCR → Int64')
print(f'  - Normalize ชื่อพรรค (Entity Resolution)')
print(f'  - ลบ outlier units (score_sum >> valid_ballots)')
print()
print(f'master_summary : {len(summary):,} → {len(s):,} rows  (ลบ {len(summary)-len(s):,} rows)')
print(f'  - แก้ type บч → บช')
print(f'  - แทน -1 ด้วย NaN และ drop แถว total_ballots=NaN')
print()
print(f'coords         : {len(coords):,} → {len(c):,} rows  (ไม่มีการลบ)')
print(f'  - ใช้ output_with_coordinates.csv แทน unit_position / unit_village')
print(f'  - พิกัดซ้ำ {len(dup_coord)} rows คงไว้ (jitter ตอนพล็อต)')
print()
print('Output files:')
for f in sorted(OUT.glob('*.csv')):
    df_tmp = pd.read_csv(f)
    print(f'  cleaned/{f.name} — {df_tmp.shape[0]:,} rows x {df_tmp.shape[1]} cols')

CLEANING REPORT
master_results : 22,422 → 19,500 rows  (ลบ 2,922 rows)
  - แก้ type บч → บช
  - ลบ province noise
  - ลบ score < 0 (missing) และ score > 99999 (OCR ต่อเลข)
  - ปัดทศนิยม OCR → Int64
  - Normalize ชื่อพรรค (Entity Resolution)
  - ลบ outlier units (score_sum >> valid_ballots)

master_summary : 710 → 707 rows  (ลบ 3 rows)
  - แก้ type บч → บช
  - แทน -1 ด้วย NaN และ drop แถว total_ballots=NaN

coords         : 342 → 342 rows  (ไม่มีการลบ)
  - ใช้ output_with_coordinates.csv แทน unit_position / unit_village
  - พิกัดซ้ำ 187 rows คงไว้ (jitter ตอนพล็อต)

Output files:
  cleaned/coords_cleaned.csv — 342 rows x 9 cols
  cleaned/master_results_cleaned.csv — 19,500 rows x 7 cols
  cleaned/master_summary_cleaned.csv — 707 rows x 10 cols


---
## 5. ทดสอบ Join cleaned data

In [44]:
r_clean = pd.read_csv(OUT / 'master_results_cleaned.csv')
s_clean = pd.read_csv(OUT / 'master_summary_cleaned.csv')
c_clean = pd.read_csv(OUT / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']

merged = (
    s_clean[s_clean['unit_number'] != -1]
    .merge(c_clean[JOIN + ['village','place','latitude','longitude']], on=JOIN, how='left')
)

has_coord = merged[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'merged shape : {merged.shape}')
print(f'มีพิกัด      : {has_coord}/{len(merged)} ({has_coord/len(merged)*100:.1f}%)')
merged.head(5)

merged shape : (681, 14)
มีพิกัด      : 681/681 (100.0%)


,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots,village,place,latitude,longitude
0,เขต,ลำปาง,งาว,นาแก,1,300.0,204.0,8.0,2.0,86.0,1,ศาลาเอนกประสงค์ บ้านทุ่งศาลา,18.784909,99.970754
1,เขต,ลำปาง,งาว,นาแก,2,700.0,404.0,27.0,17.0,252.0,2,ศาลาประชาคม วัดหนองเหียง,18.775759,99.968731
2,เขต,ลำปาง,งาว,นาแก,3,360.0,219.0,14.0,11.0,116.0,3,ศาลาวัดนาแก,18.771874,99.958053
3,เขต,ลำปาง,งาว,นาแก,4,460.0,300.0,23.0,6.0,131.0,4,ศาลากลางหมู่บ้าน บ้านแม่ฮ่าง,18.843055,99.863166
4,เขต,ลำปาง,งาว,นาแก,5,640.0,437.0,29.0,15.0,159.0,5,โรงเรียนแม่แป้นวิทยา,18.805735,99.918016
